# Week 6: Computer lab

Computer lab aims:

- To simulate geometric Brownian motion.
- To introduce numerical approximations for solutions to stochastic differential equations.

### 1 Geometric Brownian motion

Let $W \left( t \right)$ be a standard one-dimensional Brownian motion on $t \in \left[ 0, \infty \right)$. Consider the following Itô stochastic differential equation

\begin{equation*}
  dX = \alpha X dW,
\end{equation*}

where $\alpha$ is some positive real, and where $X \left( 0 \right)$ is independent of $W \left( t \right)$ for all $t \ge 0$.

We have seen that this has the solution

\begin{equation*}
  X \left( t \right) = X \left( 0 \right) \exp \left[ \alpha W \left( t \right) - \frac{1}{2} \alpha^2 t \right].
\end{equation*}

Given some positive real $\Delta t$, define $t_k = k \Delta t$ for each non-negative integer $k$. It follows that we have

\begin{equation*}
  X \left( t_{k + 1} \right) = X \left( t_k \right) \exp \left[ \alpha \Delta W_k - \frac{1}{2} \alpha^2 \Delta t \right],
\end{equation*}

where

\begin{equation*}
  \Delta W_k = W \left( t_{k + 1} \right) - W \left( t_k \right).
\end{equation*}

**Exercise 1.1**

The following code generates realizations of $W \left( t_k \right)$ for each $k \in \left\{ 0, \ldots, 1,000 \right\}$ with $\Delta t = 0.01$. By modifying this code, or otherwise, generate realizations of $X \left( t_k \right)$ for $\alpha = 1$ and $X \left( 0 \right) = 1$.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng()


def simulate_brownian(t):
    N, = t.shape
    dt = t[1] - t[0]

    W = np.zeros_like(t)
    for k in range(N - 1):
        W[k + 1] = W[k] + np.sqrt(dt) * rng.standard_normal()

    return W


def simulate_geometric_brownian(x_0, alpha, t):
    N, = t.shape
    dt = t[1] - t[0]
    X = np.zeros_like(t)
    X[0] = x_0

    for k in range(N - 1):
        X[k + 1] = X[k] * np.exp(alpha * np.sqrt(dt) * rng.standard_normal()
                                 - 0.5 * alpha * alpha * dt)
    
    return X


t = np.linspace(0.0, 10.0, 1001, dtype=np.float64)

X = simulate_geometric_brownian(1.0, 1.0, t)

plt.figure()
plt.plot(t, X)
plt.xlabel(r"$t_k$", fontsize="large")
plt.ylabel(r"$X_k$", fontsize="large")
plt.xlim(t[0], t[-1])

None

**Exercise 1.2** Generate 10,000 realizations of geometric Brownian motion, evaluated at the $t_k$, for $\alpha = 1$, $\Delta t = 0.01$, and $X \left( 0 \right) = 1$. Use these to estimate $\mathbb{E} \left[ X \left( t_k \right) \right]$ and $\mathbb{E} \left[ X^2 \left( t_k \right) \right]$. Plot the result. What do you observe?

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng()


def simulate_geometric_brownian_E(x_0, alpha, t, N_samples):
    N, = t.shape
    dt = t[1] - t[0]
    X_k = np.full(N_samples, x_0, dtype=t.dtype)

    E = np.zeros((N, 2), dtype=X_k.dtype)
    E[0, 0] = X_k.mean()
    E[0, 1] = (X_k ** 2).mean()

    for k in range(N - 1):
        X_k *= np.exp(alpha * np.sqrt(dt) * rng.standard_normal(N_samples)
                      - 0.5 * alpha * alpha * dt)
        E[k + 1, 0] = X_k.mean()
        E[k + 1, 1] = (X_k ** 2).mean()
    
    return E



t = np.linspace(0.0, 10.0, 1001, dtype=np.float64)

E = simulate_geometric_brownian_E(1.0, 1.0, t, 10000)

plt.figure()
plt.plot(t, E[:, 0])
plt.xlabel(r"$t_k$", fontsize="large")
plt.ylabel(r"$\mathbb{E} \left( X_k \right)$", fontsize="large")
plt.xlim(t[0], t[-1])

plt.figure()
plt.plot(t, E[:, 1])
plt.xlabel(r"$t_k$", fontsize="large")
plt.ylabel(r"$\mathbb{E} \left( X_k^2 \right)$", fontsize="large")
plt.xlim(t[0], t[-1])

None

*Remarks: The mean remains well-controlled and bounded, but the second moment increases with $t$.*

**Exercies 1.3** Use the matplotlib `pyplot.hist` function to plot a histogram of $X \left( 10 \right)$ for $\alpha = 1$, with a logarithmic $y$-axis scale.

- [matplotlib documentation: matplotlib.pyplot.hist](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html)

What do you observe?

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng()

x_0 = 1.0
alpha = 1.0
N_samples = 1000000
T = 10.0

X_1 = x_0 * np.exp(alpha * np.sqrt(T) * rng.standard_normal(N_samples)
                   - 0.5 * alpha * alpha * T)

plt.figure()
plt.hist(X_1, bins=50, density=True, log=True)
xlim = plt.xlim()
plt.xlim(0.0, xlim[1])

None

*Remarks: There is a low probability of very large magnitudes occuring. These large magnitude cases are rare enough that the mean remains bounded as $t \rightarrow \infty$, but frequent enough that the second moment does not.*

### 2 Ornstein-Uhlenbeck process

Let $W \left( t \right)$ be a standard one-dimensional Brownian motion on $t \in \left[ 0, \infty \right)$. Consider the following Itô stochastic differential equation

\begin{equation*}
  dX = -\gamma X dt + \sigma dW,
\end{equation*}

for some positive real $\gamma$ and some non-zero real $\sigma$, where $X \left( 0 \right)$ is independent of $W \left( t \right)$ for all $t \ge 0$.

We will later encounter the Euler-Maruyama discretization, which approximates a solution to Itô stochastic differential equations. Given some positive real $\Delta t$, define $t_k = k \Delta t$ for each non-negative integer $k$. Applying the Euler-Maruyama discretization then leads to

\begin{equation*}
  X_{k + 1} = X_k - \gamma X_k \Delta t + \sigma \sqrt{\Delta t} \xi_k,
\end{equation*}

where the $\xi_k$ are iid with $\xi_k \sim \mathcal{N} \left( 0, 1 \right)$, and where we set $X_0$ equal in distribution to $X \left( 0 \right)$, with $X_0$ independent of all later $X_k$. A justification here is that we have $\Delta W_k = W \left( t_{k + 1} \right) - W \left( t_k \right) \sim \mathcal{N} \left( 0, \Delta t \right)$, and independence of increments for a Brownian motion -- although we will investigate further details, including the errors made in this approximation, later in the course.

**Exercise 2.1** Consider $X \left( 0 \right) = 1$, $\Delta t = 0.01$, $\gamma = 4$, and $\sigma = 2$. Use the Euler-Maruyama discretization to generate realizations for the (approximated) Ornstein-Uhlenbeck process.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng()


def simulate_ou_euler_maruyama(x_0, sigma, gamma, t):
    N, = t.shape
    dt = t[1] - t[0]
    X = np.zeros_like(t)
    X[0] = x_0

    for k in range(N - 1):
        X[k + 1] = X[k] - gamma * X[k] * dt + sigma * np.sqrt(dt) * rng.standard_normal()
    
    return X


t = np.linspace(0.0, 10.0, 1001, dtype=np.float64)

X = simulate_ou_euler_maruyama(1.0, 2.0, 4.0, t)

plt.figure()
plt.plot(t, X)
plt.xlabel(r"$t_k$", fontsize="large")
plt.ylabel(r"$X_k$", fontsize="large")
plt.xlim(t[0], t[-1])

None

**Exercies 2.2** Generate 10,000 realizations for the approximation obtained with $X \left( 0 \right) = 1$, $\Delta t = 0.01$, $\gamma = 4$, and $\sigma = 2$. Use these to estimate

\begin{equation*}
  \mathbb{E} \left( X \left( t_k \right) \right),
\end{equation*}
\begin{equation*}
  \mathbb{E} \left[ \left( X \left( t_k \right) - \mathbb{E} \left( X \left( t_k \right) \right) \right)^2 \right].
\end{equation*}

Plot the results. What do you observe?

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng()


def simulate_ou_euler_maruyama_E(x_0, sigma, gamma, t, N_samples):
    N, = t.shape
    dt = t[1] - t[0]
    X_k = np.full(N_samples, x_0, dtype=t.dtype)

    E = np.zeros((N, 2), dtype=X_k.dtype)
    E[0, 0] = X_k.mean()
    E[0, 1] = ((X_k - E[0, 0]) ** 2).mean()

    for k in range(N - 1):
        X_k += - gamma * X_k * dt + sigma * np.sqrt(dt) * rng.standard_normal(N_samples)
        E[k + 1, 0] = X_k.mean()
        E[k + 1, 1] = ((X_k - E[k + 1, 0]) ** 2).mean()
    
    return E


t = np.linspace(0.0, 10.0, 1001, dtype=np.float64)

E = simulate_ou_euler_maruyama_E(1.0, 2.0, 4.0, t, 10000)

plt.figure()
plt.plot(t, E[:, 0])
plt.xlabel(r"$t_k$", fontsize="large")
plt.ylabel(r"$\mathbb{E} \left( X_k \right)$", fontsize="large")
plt.xlim(t[0], t[-1])

plt.figure()
plt.plot(t, E[:, 1])
plt.xlim(t[0], t[-1])

plt.xlabel(r"$t_k$", fontsize="large")
plt.ylabel(r"$\mathbb{V} \left( X_k \right)$", fontsize="large")

None

*Remarks: We find that the mean tends to approximately zero, and the variance tends to approximately $\sigma^2 / \left( 2 \gamma \right)$, equal to one half with the considered parameters. However experimenting with increasing $\Delta t$ reveals that we can perturb the variance, at large $t$, away from the value of one half. Increasing it further leads to instability. We will explore each of these effects later in the course.*